# Hyperparameter Tuning — Run Once
Copy the output of the last cell into `train_model.py` as `BEST_PARAMS`.

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score

sys.path.append(os.path.abspath('..'))
from src.preprocessing import clean_and_impute
from src.feature_engineering import create_features

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
df = pd.read_csv('../data/diabetes.csv')
X  = df.drop('Outcome', axis=1)
y  = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_imp, X_test_imp, _ = clean_and_impute(X_train, X_test)
X_train_final = create_features(X_train_imp)

X_res, y_res = SMOTE(random_state=42).fit_resample(X_train_final, y_train)
print(f'Shape: {df.shape} | After SMOTE: {len(y_res)} samples')

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
palette = {0: '#1f77b4', 1: '#d62728'}

for i, col in enumerate(df.columns):
    if col == 'Outcome':
        df['Outcome'].value_counts().plot(kind='bar', ax=axes[i],
            color=['#1f77b4', '#d62728'], alpha=0.8)
        axes[i].set_xticklabels(['Healthy', 'Diabetic'], rotation=0)
    else:
        sns.histplot(data=df, x=col, hue='Outcome', kde=True,
                     element='step', palette=palette, alpha=0.5, ax=axes[i])
        axes[i].legend(labels=['Diabetic', 'Healthy'], fontsize=8)
    axes[i].set_title(col, fontsize=11)

plt.suptitle('Feature Distributions: Diabetic vs Healthy', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
def objective(trial, X, y):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 500),
        'max_depth':         trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 15),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 8),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'class_weight': 'balanced',
        'random_state': 42
    }
    model = RandomForestClassifier(**params)
    return cross_val_score(model, X, y, cv=5, scoring='f1', n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: objective(trial, X_res, y_res),
               n_trials=500, show_progress_bar=True)

print(f'Best F1: {study.best_value:.4f}')
print(f'Best Params: {study.best_params}')

In [ ]:
print('BEST_PARAMS = {')
for k, v in study.best_params.items():
    val = f"'{v}'" if isinstance(v, str) else v
    print(f"    '{k}': {val},")
print("    'class_weight': 'balanced',")
print("    'random_state': 42")
print('}')
print(f'# Best F1: {study.best_value:.4f}')